# Object Detection

**AI2 · Computer Vision** — [AI Olympiad Curriculum](https://abzlitochka.github.io/ai-olympiad-curriculum)

~15 min

## Introduction

The image classification models we have dealt with so far took an image and produced a categorical result, such as the class 'number' in a MNIST problem. However, in many cases we do not want just to know that a picture portrays objects - we want to be able to determine their precise location. This is exactly the point of **object detection**.


![Object Detection](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/4-ComputerVision/11-ObjectDetection/images/Screen_Shot_2016-11-17_at_11.14.54_AM.png)

> Image from [YOLO v2 web site](https://pjreddie.com/darknet/yolov2/)



## A Naive Approach to Object Detection

Assuming we wanted to find a cat on a picture, a very naive approach to object detection would be the following:

1. Break the picture down to a number of tiles
2. Run image classification on each tile.
3. Those tiles that result in sufficiently high activation can be considered to contain the object in question.

![Naive Object Detection](ms-assets/redrawn/naive-detection.svg)

> *Image from [Exercise Notebook](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/11-ObjectDetection/ObjectDetection-TF.ipynb)*

However, this approach is far from ideal, because it only allows the algorithm to locate the object's bounding box very imprecisely. For more precise location, we need to run some sort of **regression** to predict the coordinates of bounding boxes - and for that, we need specific datasets.



## Regression for Object Detection

[This blog post](https://towardsdatascience.com/object-detection-with-neural-networks-a4e2c46b4491) has a great gentle introduction to detecting shapes.



## Datasets for Object Detection

You might run across the following datasets for this task:

* [PASCAL VOC](http://host.robots.ox.ac.uk/pascal/VOC/) - 20 classes
* [COCO](http://cocodataset.org/#home) - Common Objects in Context. 80 classes, bounding boxes and segmentation masks

![COCO](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/4-ComputerVision/11-ObjectDetection/images/coco-examples.jpg)



## Intersection over Union

While for image classification it is easy to measure how well the algorithm performs, for object detection we need to measure both the correctness of the class, as well as the precision of the inferred bounding box location. For the latter, we use the so-called **Intersection over Union** (IoU), which measures how well two boxes (or two arbitrary areas) overlap.

![IoU](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/4-ComputerVision/11-ObjectDetection/images/iou_equation.png)

> *Figure 2 from [this excellent blog post on IoU](https://pyimagesearch.com/2016/11/07/intersection-over-union-iou-for-object-detection/)*

The idea is simple - we divide the area of intersection between two figures by the area of their union. For two identical areas, IoU would be 1, while for completely disjointed areas it will be 0. Otherwise it will vary from 0 to 1. We typically only consider those bounding boxes for which IoU is over a certain value.



## Average Precision

Suppose we want to measure how well a given class of objects $C$ is recognized. To measure it, we use **Average Precision** metrics, which is calculated as follows:

1. Consider Precision-Recall curve shows the accuracy depending on a detection threshold value (from 0 to 1).
2. Depending on the threshold, we will get more or less objects detected in the image, and different values of precision and recall.
3. The curve will look like this:

![Illustration](ms-assets/en-originals/objdet-pr.png)

> *Image from [NeuroWorkshop](http://github.com/shwars/NeuroWorkshop)*

The average Precision for a given class $C$ is the area under this curve. More precisely, Recall axis is typically divided into 10 parts, and Precision is averaged over all those points:

$$
AP = {1\over11}\sum_{i=0}^{10}\mbox{Precision}(\mbox{Recall}={i\over10})
$$



## AP and IoU

We shall consider only those detections, for which IoU is above a certain value. For example, in PASCAL VOC dataset typically $\mbox{IoU Threshold} = 0.5$ is assumed, while in COCO AP is measured for different values of $\mbox{IoU Threshold}$.

![Illustration](ms-assets/en-originals/objdet-pr-iou.png)

> *Image from [NeuroWorkshop](http://github.com/shwars/NeuroWorkshop)*



## Mean Average Precision - mAP

The main metric for Object Detection is called **Mean Average Precision**, or **mAP**. It is the value of Average Precision, average across all object classes, and sometimes also over $\mbox{IoU Threshold}$. In more detail, the process of calculating **mAP** is described
[in this blog post](https://medium.com/@timothycarlen/understanding-the-map-evaluation-metric-for-object-detection-a07fe6962cf3)), and also [here with code samples](https://gist.github.com/tarlen5/008809c3decf19313de216b9208f3734).



## Different Object Detection Approaches

There are two broad classes of object detection algorithms:

* **Region Proposal Networks** (R-CNN, Fast R-CNN, Faster R-CNN). The main idea is to generate **Regions of Interests** (ROI) and run CNN over them, looking for maximum activation. It is a bit similar to the naive approach, with the exception that ROIs are generated in a more clever way. One of the majors drawbacks of such methods is that they are slow, because we need many passes of the CNN classifier over the image.
* **One-pass** (YOLO, SSD, RetinaNet) methods. In those architectures we design the network to predict both classes and ROIs in one pass.



## R-CNN: Region-Based CNN

[R-CNN](http://islab.ulsan.ac.kr/files/announcement/513/rcnn_pami.pdf) uses [Selective Search](http://www.huppelen.nl/publications/selectiveSearchDraft.pdf) to generate hierarchical structure of ROI regions, which are then passed through CNN feature extractors and SVM-classifiers to determine the object class, and linear regression to determine *bounding box* coordinates. [Official Paper](https://arxiv.org/pdf/1506.01497v1.pdf)

![RCNN](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/4-ComputerVision/11-ObjectDetection/images/rcnn1.png)

> *Image from van de Sande et al. ICCV’11*

![RCNN-1](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/4-ComputerVision/11-ObjectDetection/images/rcnn2.png)

> *Images from [this blog](https://towardsdatascience.com/r-cnn-fast-r-cnn-faster-r-cnn-yolo-object-detection-algorithms-36d53571365e)



## F-RCNN - Fast R-CNN

This approach is similar to R-CNN, but regions are defined after convolution layers have been applied.

![FRCNN](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/4-ComputerVision/11-ObjectDetection/images/f-rcnn.png)

> Image from [the Official Paper](https://www.cv-foundation.org/openaccess/content_iccv_2015/papers/Girshick_Fast_R-CNN_ICCV_2015_paper.pdf), [arXiv](https://arxiv.org/pdf/1504.08083.pdf), 2015



## Faster R-CNN

The main idea of this approach is to use neural network to predict ROIs - so-called *Region Proposal Network*. [Paper](https://arxiv.org/pdf/1506.01497.pdf), 2016

![FasterRCNN](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/4-ComputerVision/11-ObjectDetection/images/faster-rcnn.png)

> Image from [the official paper](https://arxiv.org/pdf/1506.01497.pdf)



## R-FCN: Region-Based Fully Convolutional Network

This algorithm is even faster than Faster R-CNN. The main idea is the following:

1. We extract features using ResNet-101
1. Features are processed by **Position-Sensitive Score Map**. Each object from $C$ classes is divided by $k\times k$ regions, and we are training to predict parts of objects.
1. For each part from $k\times k$ regions all networks vote for object classes, and the object class with maximum vote is selected.

![r-fcn image](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/4-ComputerVision/11-ObjectDetection/images/r-fcn.png)

> Image from [official paper](https://arxiv.org/abs/1605.06409)



## YOLO - You Only Look Once

YOLO is a realtime one-pass algorithm. The main idea is the following:

 * Image is divided into $S\times S$ regions
 * For each region, **CNN** predicts $n$ possible objects, *bounding box* coordinates and *confidence*=*probability* * IoU.

 ![YOLO](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/4-ComputerVision/11-ObjectDetection/images/yolo.png)

> Image from [official paper](https://arxiv.org/abs/1506.02640)



## Other Algorithms

* RetinaNet: [official paper](https://arxiv.org/abs/1708.02002)
   - [PyTorch Implementation in Torchvision](https://pytorch.org/vision/stable/_modules/torchvision/models/detection/retinanet.html)
   - [Keras Implementation](https://github.com/fizyr/keras-retinanet)
   - [Object Detection with RetinaNet](https://keras.io/examples/vision/retinanet/) in Keras Samples
* SSD (Single Shot Detector): [official paper](https://arxiv.org/abs/1512.02325)



## Exercises: Object Detection

Continue your learning in the following notebook:

[ObjectDetection.ipynb](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/11-ObjectDetection/ObjectDetection.ipynb)



## Conclusion

In this lesson you took a whirlwind tour of all the various ways that object detection can be accomplished!



## Challenge

Read through these articles and notebooks about YOLO and try them for yourself

* [Good blog post](https://www.analyticsvidhya.com/blog/2018/12/practical-guide-object-detection-yolo-framewor-python/) describing YOLO
 * [Official site](https://pjreddie.com/darknet/yolo/)
 * Yolo: [Keras implementation](https://github.com/experiencor/keras-yolo2), [step-by-step notebook](https://github.com/experiencor/basic-yolo-keras/blob/master/Yolo%20Step-by-Step.ipynb)
 * Yolo v2: [Keras implementation](https://github.com/experiencor/keras-yolo2), [step-by-step notebook](https://github.com/experiencor/keras-yolo2/blob/master/Yolo%20Step-by-Step.ipynb)



## Review & Self Study

* [Object Detection](https://tjmachinelearning.com/lectures/1718/obj/) by Nikhil Sardana
* [A good comparison of object detection algorithms](https://lilianweng.github.io/lil-log/2018/12/27/object-detection-part-4.html)
* [Review of Deep Learning Algorithms for Object Detection](https://medium.com/comet-app/review-of-deep-learning-algorithms-for-object-detection-c1f3d437b852)
* [A Step-by-Step Introduction to the Basic Object Detection Algorithms](https://www.analyticsvidhya.com/blog/2018/10/a-step-by-step-introduction-to-the-basic-object-detection-algorithms-part-1/)
* [Implementation of Faster R-CNN in Python for Object Detection](https://www.analyticsvidhya.com/blog/2018/11/implementation-faster-r-cnn-python-object-detection/)



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — IoU of two boxes

●○○ · ~8 min

> Intersection-over-Union (IoU) is the standard overlap measure in object detection: overlap area divided by union area. For xyxy boxes the intersection is itself a box — left = max of the lefts, right = min of the rights — and its width/height must be clamped at 0 for disjoint boxes.

Implement `iou(box, gt)` for two `[x1, y1, x2, y2]` boxes (x1 < x2, y1 < y2 guaranteed, so areas are positive) → IoU as a float in [0, 1].

<details><summary>Hint</summary>

inter = max(0, min(x2s) − max(x1s)) · max(0, min(y2s) − max(y1s)); union = areaA + areaB − inter.

</details>


In [ ]:
import numpy as np

def iou(box, gt):
    box = np.asarray(box, dtype=float)
    gt = np.asarray(gt, dtype=float)
    # your code
    return 0.0

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    def box():\n        x1 = float(rng.integers(0, 10))\n        y1 = float(rng.integers(0, 10))\n        x2 = x1 + float(rng.integers(1, 8))\n        y2 = y1 + float(rng.integers(1, 8))\n        return np.array([x1, y1, x2, y2])\n    if rng.random() < 0.5:\n        # correlate the pair so they overlap — disjoint boxes are ~72% otherwise\n        x1 = float(rng.integers(0, 8)); y1 = float(rng.integers(0, 8))\n        box = np.array([x1, y1, x1 + float(rng.integers(2, 6)), y1 + float(rng.integers(2, 6))])\n        gt = box + rng.integers(-2, 3, 4)\n        gt[:2] = np.minimum(gt[:2], box[:2] + 1)          # keep a real intersection\n        gt[2:] = np.maximum(gt[2:], gt[:2] + 1)           # stay a valid box\n        return {\"box\": box, \"gt\": gt}\n    return {\"box\": box(), \"gt\": box()}\n\ndef iou(box, gt):\n    box = np.asarray(box, dtype=float)\n    gt = np.asarray(gt, dtype=float)\n    ix1, iy1 = max(box[0], gt[0]), max(box[1], gt[1])\n    ix2, iy2 = min(box[2], gt[2]), min(box[3], gt[3])\n    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)\n    ua = (box[2] - box[0]) * (box[3] - box[1]) \\\n       + (gt[2] - gt[0]) * (gt[3] - gt[1]) - inter\n    return float(inter / ua)", "iou", 40)

### Task 2 — Pairwise IoU matrix

●●○ · ~30 min

> Matching detections to ground truth needs all pairwise IoUs — an (n, m) matrix. You can broadcast (n,1,4) against (1,m,4) or write two nested loops; both are fast enough at these sizes. Handle empty inputs gracefully: a (0, 4) array in means a (0, m) result out.

Implement `iou_matrix(boxes, gts)`: given (n, 4) and (m, 4) xyxy box arrays → the (n, m) matrix of pairwise IoU values. Either side may be empty (shape (0, 4)) — return the correctly-shaped empty matrix.

<details><summary>Hint</summary>

Loop i over boxes, j over gts, reuse the single-box IoU formula into out[i, j]. Vectorized broadcasting also works.

</details>


In [ ]:
import numpy as np

def iou_matrix(boxes, gts):
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)
    gts = np.asarray(gts, dtype=float).reshape(-1, 4)
    # your code
    return np.zeros((len(boxes), len(gts)))

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    def mk(k):\n        x1 = rng.integers(0, 10, k)\n        y1 = rng.integers(0, 10, k)\n        x2 = x1 + rng.integers(1, 8, k)\n        y2 = y1 + rng.integers(1, 8, k)\n        return np.stack([x1, y1, x2, y2], axis=1).astype(float).reshape(-1, 4)\n    return {\"boxes\": mk(int(rng.integers(0, 6))),\n            \"gts\": mk(int(rng.integers(0, 6)))}\n\ndef iou_matrix(boxes, gts):\n    a = np.asarray(boxes, dtype=float).reshape(-1, 4)\n    b = np.asarray(gts, dtype=float).reshape(-1, 4)\n    out = np.zeros((len(a), len(b)))\n    for i in range(len(a)):\n        for j in range(len(b)):\n            ix1 = max(a[i, 0], b[j, 0]); iy1 = max(a[i, 1], b[j, 1])\n            ix2 = min(a[i, 2], b[j, 2]); iy2 = min(a[i, 3], b[j, 3])\n            inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)\n            ua = ((a[i, 2] - a[i, 0]) * (a[i, 3] - a[i, 1])\n                  + (b[j, 2] - b[j, 0]) * (b[j, 3] - b[j, 1]) - inter)\n            out[i, j] = inter / ua\n    return out", "iou_matrix", 40)

### Task 3 — Non-maximum suppression

●●● · ~45 min

> NMS deduplicates overlapping detections: sort by descending score, take the best remaining box, and kill every other pending box whose IoU with it exceeds the threshold. Repeat until nothing remains. Determinism matters for grading — a stable sort makes equal scores resolve to the lower index.

Implement `nms(boxes, scores, thr)` → list of kept indices (ints) in selection order (descending score). Greedy: pick the highest-scored remaining box, keep it, and remove every remaining box with IoU > thr to it. Score ties resolve to the lower index first (stable sort). `boxes` may be empty (shape (0, 4)).

<details><summary>Hint</summary>

order = list(np.argsort(-scores, kind='stable')); pop(0) each round into keep, then retain only boxes with IoU ≤ thr.

</details>


In [ ]:
import numpy as np

def nms(boxes, scores, thr):
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)
    scores = np.asarray(scores, dtype=float)
    # your code — keep highest-score box, suppress IoU-overlapping rest
    return [-1]  # wrong — replace

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n = int(rng.integers(0, 9))\n    x1 = rng.integers(0, 8, n)\n    y1 = rng.integers(0, 8, n)\n    x2 = x1 + rng.integers(1, 6, n)\n    y2 = y1 + rng.integers(1, 6, n)\n    boxes = np.stack([x1, y1, x2, y2], axis=1).astype(float).reshape(-1, 4)\n    return {\"boxes\": boxes,\n            \"scores\": np.round(rng.random(n), 1),   # quantized: score ties exercise the stable-sort rule\n            \"thr\": float(rng.choice([0.2, 0.3, 0.5, 0.7]))}\n\ndef nms(boxes, scores, thr):\n    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)\n    scores = np.asarray(scores, dtype=float)\n    order = [int(i) for i in np.argsort(-scores, kind=\"stable\")]\n    keep = []\n    while order:\n        i = order.pop(0)\n        keep.append(i)\n        rest = []\n        for j in order:\n            b, g = boxes[i], boxes[j]\n            ix1 = max(b[0], g[0]); iy1 = max(b[1], g[1])\n            ix2 = min(b[2], g[2]); iy2 = min(b[3], g[3])\n            inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)\n            ua = ((b[2] - b[0]) * (b[3] - b[1])\n                  + (g[2] - g[0]) * (g[3] - g[1]) - inter)\n            if inter / ua <= thr:\n                rest.append(j)\n        order = rest\n    return keep", "nms", 40)

### Task 4 — TP / FP / FN by IoU matching

●●● · ~60 min

> Detection metrics reduce to counting TP, FP and FN via greedy matching: process predictions in descending-score order; each prediction claims the still-unmatched ground-truth box with the highest IoU. If that best IoU is ≥ threshold it is a true positive and the GT is consumed, otherwise it is a false positive. GTs never claimed are false negatives.

Implement `count_tfpn(pred_boxes, scores, gt_boxes, thr)` → `{'tp': int, 'fp': int, 'fn': int}`. Match greedily in descending-score order (score ties → lower index); each prediction takes the unused GT with maximum IoU (IoU ties → lowest GT index); IoU ≥ thr → TP and the GT is consumed, else FP. Unmatched GTs count as FN. Either side may be empty (shape (0, 4)).

<details><summary>Hint</summary>

Track a used[] mask over GTs; for each pred (in argsort order) scan all unused GTs for the max IoU, then compare with thr.

</details>


In [ ]:
import numpy as np

def count_tfpn(pred_boxes, scores, gt_boxes, thr):
    pred_boxes = np.asarray(pred_boxes, dtype=float).reshape(-1, 4)
    scores = np.asarray(scores, dtype=float)
    gt_boxes = np.asarray(gt_boxes, dtype=float).reshape(-1, 4)
    # your code — greedy match by descending score
    return {"tp": -1, "fp": -1, "fn": -1}  # wrong — replace

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n, m = int(rng.integers(0, 7)), int(rng.integers(0, 7))\n    # cluster boxes in a small patch so each pred overlaps several GTs —\n    # matching to an arbitrary/first GT instead of the best-IoU one must fail\n    x0, y0 = int(rng.integers(0, 4)), int(rng.integers(0, 4))\n    def mk(k):\n        x1 = x0 + rng.integers(0, 3, k)\n        y1 = y0 + rng.integers(0, 3, k)\n        x2 = x1 + rng.integers(2, 6, k)\n        y2 = y1 + rng.integers(2, 6, k)\n        return np.stack([x1, y1, x2, y2], axis=1).astype(float).reshape(-1, 4)\n    return {\"pred_boxes\": mk(n),\n            \"scores\": np.round(rng.random(n), 1),   # rounded → score ties exercise the index tie-break\n            \"gt_boxes\": mk(m),\n            \"thr\": float(rng.choice([0.3, 0.5, 0.7]))}\n\ndef count_tfpn(pred_boxes, scores, gt_boxes, thr):\n    P = np.asarray(pred_boxes, dtype=float).reshape(-1, 4)\n    G = np.asarray(gt_boxes, dtype=float).reshape(-1, 4)\n    order = [int(i) for i in np.argsort(-np.asarray(scores, dtype=float), kind=\"stable\")]\n    used = np.zeros(len(G), dtype=bool)\n    tp = fp = 0\n    for i in order:\n        best, bj = 0.0, -1\n        for j in range(len(G)):\n            if used[j]:\n                continue\n            p, g = P[i], G[j]\n            ix1 = max(p[0], g[0]); iy1 = max(p[1], g[1])\n            ix2 = min(p[2], g[2]); iy2 = min(p[3], g[3])\n            inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)\n            ua = ((p[2] - p[0]) * (p[3] - p[1])\n                  + (g[2] - g[0]) * (g[3] - g[1]) - inter)\n            v = inter / ua\n            if v > best:\n                best, bj = v, j\n        if bj >= 0 and best >= thr:\n            used[bj] = True\n            tp += 1\n        else:\n            fp += 1\n    return {\"tp\": tp, \"fp\": fp, \"fn\": int((~used).sum())}", "count_tfpn", 40)

### Task 5 — Average Precision (all-point)

●●● · ~45 min

> Average Precision compresses a single-class precision–recall curve into one number — mAP is just its mean over classes. Sort detections by descending score, accumulate TP/FP down the ranked list, and each rank gives a (recall, precision) point. The interpolated precision at recall r is the maximum precision among all points with recall ≥ r — the 'envelope' that only decreases. All-point AP (VOC-style) integrates it exactly: AP = Σ (R_k − R_{k−1}) · p_interp(R_k) over every rank where recall increases.

Implement `average_precision(scores, tp, npos)` for one class: `scores` (n,) floats, `tp` (n,) 0/1 flags marking correct detections, `npos` = total ground-truth objects (npos ≥ number of TPs). Sort by descending score (ties → lower index first, stable sort), compute cumulative TP/FP, recall = cumTP/npos, precision = cumTP/(cumTP+cumFP). Apply the precision envelope (each point takes the max precision at recall ≥ its own), then AP = Σ over ranks where recall increases of (Δrecall · envelope precision). Return 0.0 if `npos ≤ 0` or there are no detections.

<details><summary>Hint</summary>

Prepend (recall=0, prec=0) sentinels; sweep the precision array right-to-left taking running max for the envelope; then sum (mrec[i+1] − mrec[i]) · mpre[i+1] wherever mrec[i+1] != mrec[i].

</details>


In [ ]:
import numpy as np

def average_precision(scores, tp, npos):
    scores = np.asarray(scores, dtype=float)
    tp = np.asarray(tp, dtype=float)
    # your code
    return 0.0

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n = int(rng.integers(0, 11))\n    tp = rng.integers(0, 2, n)\n    npos = int(tp.sum()) + int(rng.integers(0, 4))\n    return {\"scores\": np.round(rng.random(n), 1),   # quantized: ties exercise the stable-sort rule\n            \"tp\": tp,\n            \"npos\": npos}\n\ndef average_precision(scores, tp, npos):\n    scores = np.asarray(scores, dtype=float)\n    tp = np.asarray(tp, dtype=float)\n    if npos <= 0 or tp.size == 0:\n        return 0.0\n    order = np.argsort(-scores, kind=\"stable\")\n    tp = tp[order]\n    fp = 1.0 - tp\n    ctp, cfp = np.cumsum(tp), np.cumsum(fp)\n    rec = ctp / float(npos)\n    prec = ctp / np.maximum(ctp + cfp, 1e-12)\n    mrec = np.concatenate([[0.0], rec])\n    mpre = np.concatenate([[0.0], prec])\n    for i in range(len(mpre) - 2, -1, -1):\n        mpre[i] = max(mpre[i], mpre[i + 1])\n    idx = np.nonzero(mrec[1:] != mrec[:-1])[0]\n    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))", "average_precision", 40)

---

**Go deeper**
- [Original lesson (Microsoft, MIT)](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/11-ObjectDetection)
- [Yandex ML handbook (RU): convolutional networks](https://contest.yandex.ru/tracks/ml/deep-learning-architectures/convolutional-neural-networks)